In [1]:
import torch
import torch.nn as nn
import math
import random

random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


**vocabulary**

In [2]:
special_tokens = ["<pad>", "<bos>", "<eos>"]
base_words = ["one", "two", "three", "four", "five", "six", "seven", "eight", "nine"]

vocab = special_tokens + base_words
stoi = {token: idx for idx, token in enumerate(vocab)}
itos = {idx: token for token, idx in stoi.items()}

In [3]:
PAD_IDX = stoi["<pad>"]
BOS_IDX = stoi["<bos>"]
EOS_IDX = stoi["<eos>"]

vocab_size = len(vocab)
print("Vocab size :", vocab_size)
print(stoi)

Vocab size : 12
{'<pad>': 0, '<bos>': 1, '<eos>': 2, 'one': 3, 'two': 4, 'three': 5, 'four': 6, 'five': 7, 'six': 8, 'seven': 9, 'eight': 10, 'nine': 11}


**Encode / decode helpers**

In [4]:
def encode_tokens(tokens):
    return [stoi[t] for t in tokens]

def decode_tokens(ids):
    words = []
    for idx in ids:
        token = itos[int(idx)]
        if token == "<eos>":
            break
        if token not in ["<pad>", "<bos>"]:
            words.append(token)
    return words

**Create toy dataset**

In [5]:
def generate_sample(min_len=3, max_len=5):
    length = random.randint(min_len, max_len)
    src_words = random.sample(base_words, length)
    tgt_words = list(reversed(src_words))
    return src_words, tgt_words

train_pairs = [generate_sample() for _ in range(1200)]
test_pairs = [generate_sample() for _ in range(200)]

print(train_pairs[:5])

[(['two', 'one', 'six', 'three', 'nine'], ['nine', 'three', 'six', 'one', 'two']), (['three', 'two', 'six'], ['six', 'two', 'three']), (['nine', 'two', 'five', 'four', 'one'], ['one', 'four', 'five', 'two', 'nine']), (['two', 'four', 'nine'], ['nine', 'four', 'two']), (['one', 'four', 'six', 'seven', 'five'], ['five', 'seven', 'six', 'four', 'one'])]


**Batch preparation**

In [6]:
def pad_sequences(sequences, pad_value=PAD_IDX):
    max_len = max(len(seq) for seq in sequences)
    padded = []
    for seq in sequences:
        padded.append(seq + [pad_value] * (max_len - len(seq)))
    
    return torch.tensor(padded, dtype=torch.long)

In [7]:
def make_batch(pairs):
    src_batch = [] # encoder input
    tgt_input_batch = [] # decoder input
    tgt_output_batch = [] # decoder expected correct answer
    
    for src_words, tgt_words in pairs:
        src_ids = encode_tokens(src_words) + [EOS_IDX]
        tgt_input_ids = [BOS_IDX] + encode_tokens(tgt_words)
        tgt_output_ids = encode_tokens(tgt_words) + [EOS_IDX]
        
        src_batch.append(src_ids)
        tgt_input_batch.append(tgt_input_ids)
        tgt_output_batch.append(tgt_output_ids)

    src_batch = pad_sequences(src_batch)
    tgt_input_batch = pad_sequences(tgt_input_batch)
    tgt_output_batch = pad_sequences(tgt_output_batch)
    
    return src_batch, tgt_input_batch, tgt_output_batch

**Positional Encoding**

In [8]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model)
        )
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        self.register_buffer("pe", pe.unsqueeze(0))
    
    def forward(self, x):
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len]

**Masks**

In [9]:
def mask_src_mask(src):
    return (src != PAD_IDX).unsqueeze(1).unsqueeze(2)

def make_tgt_mask(tgt):
    batch_size, tgt_len = tgt.shape
    
    pad_mask = (tgt != PAD_IDX).unsqueeze(1).unsqueeze(2)
    causal_mask = torch.tril(torch.ones((tgt_len, tgt_len), device=tgt.device)).bool()
    causal_mask = causal_mask.unsqueeze(0).unsqueeze(0)
    
    return pad_mask & causal_mask

**Multi-Head Attention**

In [10]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        
    def forward(self, q, k, v, mask=None):
        batch_size = q.size(0)
        
        Q_len = q.size(1)
        K_len = k.size(1)
        V_len = v.size(1)
        
        Q = self.q_proj(q)
        K = self.k_proj(k)
        V = self.v_proj(v)
        
        Q = Q.view(batch_size, Q_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, K_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, V_len, self.num_heads, self.head_dim).transpose(1, 2)
        
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        
        if mask is not None:
            scores = scores.masked_fill(mask==0, self.float("-inf"))
            
        attention = torch.softmax(attention, dim=-1)
        output = torch.matmul(attention, V)
        
        output = output.transpose(1, 2).contiguous().view(batch_size, Q_len, self.d_model)
        output = self.out_proj(output)
        
        return output, attention

**Feed Forward Network**